# 02a — Retrain the 3 CNNs on Clean Split + Metrics

Runs **ConvNeXt-Tiny, Inception V3, and the Vanilla CNN** on the clean 70/15/15 split, with augmentation. Each saves its weights (`*_clean.pth`) and predictions (`*_preds.npz`) to `/kaggle/working/` and prints its full metric block.

These three are fast, so they comfortably finish in one session. Transformers are in **02b**. The combined comparison table is built in **02c** from the saved `.npz` files.

**Setup:** attach (1) the dataset and (2) notebook 01's output; set `SPLIT_DIR` below.

In [1]:
import os
for r, d, f in os.walk('/kaggle/input'):
    if any(x.endswith('.npy') for x in f):
        print(r, '->', [x for x in f if x.endswith('.npy')])

/kaggle/input/notebooks/tochyokafor/build-clean-split -> ['clean_train_indices.npy', 'clean_test_indices.npy', 'clean_val_indices.npy']


In [2]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
# Path where notebook 01's .npy outputs are mounted (adjust to your dataset name):
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"  # <-- set to notebook 01 output dataset
OUT_DIR   = "/kaggle/working"
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f'Missing path: {pth}'
print('Paths OK')

Paths OK


In [3]:
!pip install timm --quiet
import numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, roc_auc_score)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# --- load the clean split (single source of truth) ---
train_idx = np.load(f'{SPLIT_DIR}/clean_train_indices.npy').tolist()
val_idx   = np.load(f'{SPLIT_DIR}/clean_val_indices.npy').tolist()
test_idx  = np.load(f'{SPLIT_DIR}/clean_test_indices.npy').tolist()
class_names = open(f'{SPLIT_DIR}/class_names.txt').read().splitlines()
print(f'Train {len(train_idx)}  Val {len(val_idx)}  Test {len(test_idx)}')
print('Classes:', class_names)
SEVERE_IDX = class_names.index('Severe')
print('Severe class index:', SEVERE_IDX)

Device: cuda
Train 2485  Val 533  Test 536
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Severe class index: 4


In [4]:
# ============================================================
# Shared transforms + loader factory
# Augmentation ON for training (defensible on the smaller clean set).
# `size` varies per model (299 Inception, 224 the rest).
# ============================================================
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM,
    ])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, crop_from=None):
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  val_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True,  num_workers=2),
            DataLoader(val_ds,   BATCH, shuffle=False, num_workers=2),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=2))

In [5]:
# ============================================================
# Shared train / evaluate / save helpers
# ============================================================
RESULTS = {}  # model_name -> metrics dict

def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        running = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):        # Inception aux logits
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,'logits') else out, y)
            loss.backward(); optimizer.step()
            running += loss.item()
        if scheduler: scheduler.step()
        print(f'  epoch {ep+1}/{epochs}  loss {running/len(loader):.4f}')
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        out = out.logits if hasattr(out,'logits') else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def report_and_save(name, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weap  = f1_score(labels, preds, average='weighted')
    pr, rc, f1, sup = precision_recall_fscore_support(labels, preds, labels=range(NUM_CLASSES), zero_division=0)
    try:
        auc_macro = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
        auc_per = roc_auc_score(np.eye(NUM_CLASSES)[labels], probs, multi_class='ovr', average=None)
    except Exception as e:
        print('  AUC warning:', e); auc_macro, auc_per = float('nan'), [float('nan')]*NUM_CLASSES
    print(f'\n=== {name} ===')
    print(f'Accuracy {acc:.4f} | Macro-F1 {f1_macro:.4f} | Weighted-F1 {f1_weap:.4f} | Macro-AUC {auc_macro:.4f}')
    print(f"{'Class':<16}{'Prec':>7}{'Rec':>7}{'F1':>7}{'AUC':>7}{'N':>6}")
    for c, cn in enumerate(class_names):
        star = '  <-- SEVERE' if c == SEVERE_IDX else ''
        print(f'{cn:<16}{pr[c]:>7.3f}{rc[c]:>7.3f}{f1[c]:>7.3f}{auc_per[c]:>7.3f}{sup[c]:>6}{star}')
    # save weights already done by caller; save preds here
    np.savez(f'{OUT_DIR}/{name}_preds.npz', probs=probs, preds=preds, labels=labels)
    RESULTS[name] = dict(acc=acc, f1_macro=f1_macro, f1_weighted=f1_weap,
                         auc_macro=auc_macro, severe_recall=rc[SEVERE_IDX],
                         severe_f1=f1[SEVERE_IDX])
    print(f'  saved {OUT_DIR}/{name}_preds.npz')

## Model 3 — ConvNeXt-Tiny. torchvision, classifier[2], Adam 1e-4.

In [6]:
tr, va, te = make_loaders(224)
cnx = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
cnx.classifier[2] = nn.Linear(cnx.classifier[2].in_features, NUM_CLASSES)
cnx = cnx.to(device)
crit = nn.CrossEntropyLoss()
opt  = optim.Adam(cnx.parameters(), lr=1e-4)
print('Training ConvNeXt-Tiny...')
cnx = train_model(cnx, tr, crit, opt)
torch.save(cnx.state_dict(), f'{OUT_DIR}/convnext_tiny_clean.pth')
probs, labels = evaluate(cnx, te)
report_and_save('convnext_tiny', probs, labels)
del cnx; torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 210MB/s]


Training ConvNeXt-Tiny...
  epoch 1/10  loss 0.2459
  epoch 2/10  loss 0.0887
  epoch 3/10  loss 0.0829
  epoch 4/10  loss 0.0689
  epoch 5/10  loss 0.0492
  epoch 6/10  loss 0.0514
  epoch 7/10  loss 0.0519
  epoch 8/10  loss 0.0493
  epoch 9/10  loss 0.0262
  epoch 10/10  loss 0.0503

=== convnext_tiny ===
Accuracy 0.9235 | Macro-F1 0.9050 | Weighted-F1 0.9229 | Macro-AUC 0.9968
Class              Prec    Rec     F1    AUC     N
Mild              0.919  1.000  0.958  1.000    79
Moderate          0.702  0.983  0.819  0.994    60
No_DR             0.993  1.000  0.997  1.000   146
Proliferate_DR    0.950  0.921  0.935  0.992   164
Severe            1.000  0.690  0.816  0.998    87  <-- SEVERE
  saved /kaggle/working/convnext_tiny_preds.npz


## Model 4 — Inception V3. torchvision, fc + AuxLogits.fc, Adam 1e-4, aux loss. Needs 299px + aux handling.

In [7]:
tr, va, te = make_loaders(299, crop_from=320)
inc = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)
inc.fc = nn.Linear(inc.fc.in_features, NUM_CLASSES)
inc.AuxLogits.fc = nn.Linear(inc.AuxLogits.fc.in_features, NUM_CLASSES)
inc = inc.to(device)
crit = nn.CrossEntropyLoss()
opt  = optim.Adam(inc.parameters(), lr=1e-4)
print('Training Inception V3...')
inc = train_model(inc, tr, crit, opt, aux=True)  # aux=True handles the tuple output in train mode
torch.save(inc.state_dict(), f'{OUT_DIR}/inception_v3_clean.pth')
probs, labels = evaluate(inc, te)   # eval mode returns single output
report_and_save('inception_v3', probs, labels)
del inc; torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 207MB/s] 


Training Inception V3...
  epoch 1/10  loss 0.4243
  epoch 2/10  loss 0.1984
  epoch 3/10  loss 0.1411
  epoch 4/10  loss 0.1196
  epoch 5/10  loss 0.0956
  epoch 6/10  loss 0.0780
  epoch 7/10  loss 0.0624
  epoch 8/10  loss 0.0563
  epoch 9/10  loss 0.0789
  epoch 10/10  loss 0.0590

=== inception_v3 ===
Accuracy 0.9179 | Macro-F1 0.9217 | Weighted-F1 0.9199 | Macro-AUC 0.9989
Class              Prec    Rec     F1    AUC     N
Mild              0.669  1.000  0.802  1.000    79
Moderate          0.967  0.983  0.975  0.999    60
No_DR             0.993  1.000  0.997  1.000   146
Proliferate_DR    0.992  0.756  0.858  0.995   164
Severe            0.988  0.966  0.977  1.000    87  <-- SEVERE
  saved /kaggle/working/inception_v3_preds.npz


## Model 5 — CNN Vanilla. Custom architecture from scratch, Adam 1e-3. (Re-define to match your original.)

In [8]:
# Vanilla CNN — YOUR original architecture, adjusted only for 224px input.
# Original design preserved exactly: conv(3->32->64->128->256), 4x maxpool,
# flatten -> fc1(256) -> dropout(0.25) -> fc2(5).
# Only change vs original: input 512->224 so flatten = 256*14*14 = 50176
# (was 256*32*32 = 262144 at 512px). Now matches the other 4 models' preprocessing.
class VanillaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.relu    = nn.ReLU()
        self.maxPool = nn.MaxPool2d(kernel_size=2)
        self.dropout = nn.Dropout(p=0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 14 * 14, 256)   # 224 -> 112 -> 56 -> 28 -> 14
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.relu(self.maxPool(self.conv1(x)))
        x = self.relu(self.maxPool(self.conv2(x)))
        x = self.relu(self.maxPool(self.conv3(x)))
        x = self.relu(self.maxPool(self.conv4(x)))
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

tr, va, te = make_loaders(224)
vcnn = VanillaCNN().to(device)
crit = nn.CrossEntropyLoss()
opt  = optim.Adam(vcnn.parameters(), lr=1e-3)   # 1e-3 for from-scratch (your original)
print('Training Vanilla CNN...')
vcnn = train_model(vcnn, tr, crit, opt)
torch.save(vcnn.state_dict(), f'{OUT_DIR}/vanilla_cnn_clean.pth')
probs, labels = evaluate(vcnn, te)
report_and_save('vanilla_cnn', probs, labels)
del vcnn; torch.cuda.empty_cache()

Training Vanilla CNN...
  epoch 1/10  loss 0.9174
  epoch 2/10  loss 0.6465
  epoch 3/10  loss 0.4892
  epoch 4/10  loss 0.4016
  epoch 5/10  loss 0.3408
  epoch 6/10  loss 0.3258
  epoch 7/10  loss 0.2864
  epoch 8/10  loss 0.2702
  epoch 9/10  loss 0.2619
  epoch 10/10  loss 0.2528

=== vanilla_cnn ===
Accuracy 0.7687 | Macro-F1 0.6691 | Weighted-F1 0.7406 | Macro-AUC 0.9850
Class              Prec    Rec     F1    AUC     N
Mild              0.664  1.000  0.798  1.000    79
Moderate          0.382  0.833  0.524  0.937    60
No_DR             0.993  1.000  0.997  1.000   146
Proliferate_DR    0.985  0.793  0.878  0.989   164
Severe            1.000  0.080  0.149  0.999    87  <-- SEVERE
  saved /kaggle/working/vanilla_cnn_preds.npz


### Per-session mini-summary (these 3 models)

In [9]:
import pandas as pd
df = pd.DataFrame(RESULTS).T
print('Models trained this session:', list(RESULTS.keys()))
print(df.round(4).to_string())
print('\nWeights + .npz saved to /kaggle/working/. Now Save Version (Commit),')
print('then run 02b for the transformers.')

Models trained this session: ['convnext_tiny', 'inception_v3', 'vanilla_cnn']
                  acc  f1_macro  f1_weighted  auc_macro  severe_recall  severe_f1
convnext_tiny  0.9235    0.9050       0.9229     0.9968         0.6897     0.8163
inception_v3   0.9179    0.9217       0.9199     0.9989         0.9655     0.9767
vanilla_cnn    0.7687    0.6691       0.7406     0.9850         0.0805     0.1489

Weights + .npz saved to /kaggle/working/. Now Save Version (Commit),
then run 02b for the transformers.
